In [1]:
# Import Models

import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import NearestNeighbors

In [2]:
df_X2 = pd.read_csv("homework_2.1.csv")

In [3]:
df_X2.head()

,time,G1,G2,G3
0,0,0.882026,1.441575,0.065409
1,1,0.210079,-0.163880,0.140310
2,2,0.509369,-0.115242,0.819830
3,3,1.150447,1.014698,0.607632
4,4,0.973779,-0.046562,0.610066


In [15]:
import pandas as pd
import statsmodels.formula.api as smf

# 2. Run the regression specifically for Group 1 (G1)
model = smf.ols('G1 ~ time', data=df_X2).fit()

# 3. Print the coefficient rounded to 5 decimal places
print("Group 1 Time Coefficient:", round(model.params['time'], 5))

Group 1 Time Coefficient: 0.0085


In [14]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

df_long = pd.melt(df_X2, id_vars=['time'], value_vars=['G1', 'G2', 'G3'], var_name='group', value_name='outcome')

# Test with an explicit intercept (Group 1 becomes the reference baseline intercept)
model_with_intercept = smf.ols('outcome ~ time + C(group)', data=df_long).fit()
print(model_with_intercept.summary())

                            OLS Regression Results                            
Dep. Variable:                outcome   R-squared:                       0.311
Model:                            OLS   Adj. R-squared:                  0.304
Method:                 Least Squares   F-statistic:                     44.55
Date:                Fri, 29 May 2026   Prob (F-statistic):           8.72e-24
Time:                        03:07:11   Log-Likelihood:                -216.89
No. Observations:                 300   AIC:                             441.8
Df Residuals:                     296   BIC:                             456.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          0.0786      0.071      1.

In [22]:
df_X22 = pd.read_csv('homework_2.2.csv')

In [19]:

# Calculate the mean outcome for treated (X = 1) and untreated (X = 0) populations
mean_treated = df_X22[df_X22['X'] == 1]['Y'].mean()
mean_untreated = df_X22[df_X22['X'] == 0]['Y'].mean()

# Calculate the simple treatment effect
mean_effect = mean_treated - mean_untreated

print(f"Mean outcome for treated (X=1): {mean_treated:.4f}")
print(f"Mean outcome for untreated (X=0): {mean_untreated:.4f}")
print(f"Simple Mean Effect: {mean_effect:.4f}")

Mean outcome for treated (X=1): 7.8428
Mean outcome for untreated (X=0): 4.9220
Simple Mean Effect: 2.9207


In [20]:
# Set a random seed for reproducibility
np.random.seed(42)

# Number of bootstrap replications
B = 10000
boot_effects = []

for _ in range(B):
    # Sample rows with replacement
    boot_df = df_X22.sample(frac=1.0, replace=True)
    
    # Calculate the simple treatment effect for the bootstrap sample
    mean_treated = boot_df[boot_df['X'] == 1]['Y'].mean()
    mean_untreated = boot_df[boot_df['X'] == 0]['Y'].mean()
    
    boot_effects.append(mean_treated - mean_untreated)

# Calculate the variance of the bootstrap treatment effects
bootstrap_variance = np.var(boot_effects, ddof=1)
print(f"Bootstrap Variance: {bootstrap_variance:.4f}")

Bootstrap Variance: 0.0326


In [23]:
from scipy.stats import skew

# Set random seed for reproducibility
np.random.seed(42)

# Number of bootstrap replications
B = 5000
boot_effects = []

for _ in range(B):
    # Sample rows with replacement
    boot_df = df_X22.sample(frac=1.0, replace=True)
    
    # Reshape variables for linear regression
    X_boot = boot_df[['X']]
    y_boot = boot_df['Y']
    
    # Fit linear regression model with intercept
    model = LinearRegression().fit(X_boot, y_boot)
    
    # The coefficient of X is the measured treatment effect
    boot_effects.append(model.coef_[0])

# Compute the skewness of the measured effects
effect_skewness = skew(boot_effects)
print(f"Skewness of the effect measured: {effect_skewness:.4f}")

Skewness of the effect measured: 0.0415


In [ ]:
def run_pareto_bootstrap(shape_alpha=3.0, scale_xm=1.0, num_bootstraps=1000):
    sample_sizes = [50, 100, 500, 1000, 5000]    # Define different sample sizes
    results = {}
    
    for n in sample_sizes:
        original_sample = (np.random.pareto(shape_alpha, size=n) + 1) * scale_xm
        
        bootstrap_means = []
        for _ in range(num_bootstraps):
            boot_sample = np.random.choice(original_sample, size=n, replace=True)
            bootstrap_means.append(np.mean(boot_sample))
            
        var_of_mean = np.var(bootstrap_means)
        results[n] = var_of_mean
        
        print(f"Full Sample Size (n) = {n:4d} | Variance of the Mean = {var_of_mean:.6f}")
        
    return results

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Load your data
df = pd.read_csv("homework_3.1.csv")

# Define your event time
t_0 = 50  

# Define Baseline terms
df["t"] = df["time"]
df["t_sq"] = df["time"] ** 2

# Set Event indicator
df["D"] = (df["time"] >= t_0).astype(int)

# Set Interaction terms
df["interaction_slope"] = (df["time"] - t_0) * df["D"]
df["interaction_curvature"] = ((df["time"] - t_0) ** 2) * df["D"]

# Define Independent Variables
X = df[["t", "t_sq", "D", "interaction_slope", "interaction_curvature"]]
X = sm.add_constant(X)

# Fit the model for one of your variables
model = sm.OLS(df["value1"], X).fit()

# View the results